# Step 3: Defect Analysis (03_Defect_Analysis)

This notebook researches the subtraction, thresholding, and morphological cleaning pipeline to isolate sticker-level defects.

We use a **solved cube with a single defect** (`inspect_solved_single_defect.jpg`) as a case study to demonstrate how morphological operators filter background noise and structure the final defect contours.

In [ ]:
%load_ext autoreload
%autoreload 2
import cv2
import numpy as np
import matplotlib.pyplot as plt
import core_functions as cf

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load and Align the Image
We load the solved reference and solved single defect inspection image, and align them.

In [ ]:
ref_img = cf.load_image('data/solved_pipeline/ref_solved.jpg')
inspect_img = cf.load_image('data/solved_pipeline/inspect_solved_single_defect.jpg')

aligned_img, _, _, _, _, _ = cf.align_images_orb(inspect_img, ref_img)
print("Alignment complete!")

## 2. Absolute Difference in brightness (HSV V-Channel)
To isolate defects regardless of shadows, we convert the images to HSV, extract the V (Value/Brightness) channel, and compute the absolute difference. This highlights differences in colors/stickers.

In [ ]:
ref_hsv = cv2.cvtColor(ref_img, cv2.COLOR_RGB2HSV)
aligned_hsv = cv2.cvtColor(aligned_img, cv2.COLOR_RGB2HSV)

ref_v = ref_hsv[:, :, 2]
aligned_v = aligned_hsv[:, :, 2]

diff = cv2.absdiff(ref_v, aligned_v)

plt.imshow(diff, cmap='gray')
plt.title("Value (V) Channel Absolute Difference")
plt.axis('off')
plt.show()

## 3. Binarization (Thresholding)
We apply binary thresholding to isolate significant intensity changes from minor noise. Notice that the resulting raw mask contains edge alignment artifacts (thin lines) and small spots.

In [ ]:
threshold_value = 35
_, thresh = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)

plt.imshow(thresh, cmap='gray')
plt.title(f"Raw Binary Mask (Threshold = {threshold_value})")
plt.axis('off')
plt.show()

## 4. Morphological Cleaning: Opening & Closing
- **Morphological Opening (Erosion followed by Dilation):** Eliminates isolated white pixels and thin edge alignment lines (edge noise).
- **Morphological Closing (Dilation followed by Erosion):** Bridges gaps and fills holes within the defect region, creating a single contiguous mask.

In [ ]:
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
ax1.imshow(opened, cmap='gray')
ax1.set_title("1. After Morphological Opening (Thin edge lines and spots removed)")
ax1.axis('off')

ax2.imshow(closed, cmap='gray')
ax2.set_title("2. After Morphological Closing (Defect interior filled and smoothed)")
ax2.axis('off')
plt.show()

## Conclusion
Using morphological opening and closing with a rectangular kernel size of $9\times9$ pixels successfully purifies the defect mask from registration noise and bridges split fragments, preparing it for contour contouring and bounding-box classification in the final pipeline.